# Reasoning-distillation data generator (laptop **or** Colab)

Generates verified concise CoT math solutions with **DeepSeek V4 Flash via OpenRouter**,
keeps only answer-verified traces, and writes them as the `reasoning_distill` SFT source.

Auto-detects environment:
- **Colab** → mounts Drive, clones the repo, reads `OPENROUTER_API_KEY` from Colab Secrets.
- **Laptop** → uses your local repo + `SYNAPSE_DIR` + repo `.env`.

Run the **calibration** cell first (3k problems → prints yield% + projected full cost), then the full run.

In [ ]:
# 1. Detect environment + set SYNAPSE_DIR
import os
try:
    from google.colab import drive
    COLAB = True
    drive.mount('/content/drive', force_remount=False)
    SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
except ImportError:
    COLAB = False
    SYNAPSE_DIR = os.environ.get('SYNAPSE_DIR') or os.path.abspath('./synapse')
os.environ['SYNAPSE_DIR'] = SYNAPSE_DIR
print('COLAB =', COLAB, '| SYNAPSE_DIR =', SYNAPSE_DIR)

In [ ]:
# 2. Deps
!pip install -q openai datasets sympy python-dotenv tqdm

In [ ]:
# 3. Locate the repo (clone on Colab; find it locally on laptop)
import os, subprocess
if COLAB:
    REPO_DIR = '/content/synapse_repo'
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
    else:
        subprocess.run(['git','clone','--depth=1','https://github.com/ajencinas/synapse.git',REPO_DIR], check=True)
else:
    d = os.path.abspath('.')
    while d != os.path.dirname(d) and not os.path.isfile(os.path.join(d,'sft','generate_reasoning_distill.py')):
        d = os.path.dirname(d)
    REPO_DIR = d
assert os.path.isfile(os.path.join(REPO_DIR,'sft','generate_reasoning_distill.py')), \
    'generate_reasoning_distill.py not found — on laptop, run this notebook from inside the repo'
print('REPO_DIR =', REPO_DIR)

In [ ]:
# 4. API key — works with EITHER an OpenRouter or a direct DeepSeek key.
#    Colab: add OPENROUTER_API_KEY *or* DEEPSEEK_API_KEY in Secrets (🔑).
#    Laptop: read automatically from repo .env. The generator auto-detects which.
import os
for name in ('OPENROUTER_API_KEY', 'DEEPSEEK_API_KEY'):
    if not os.environ.get(name) and COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v: os.environ[name] = v
        except Exception:
            pass
have = [n for n in ('OPENROUTER_API_KEY','DEEPSEEK_API_KEY') if os.environ.get(n)]
print('keys in env:', have or 'none yet — will fall back to repo .env')
print('provider: DeepSeek direct is preferred when both keys are present')

In [ ]:
# 5. CALIBRATION — 3k problems. Prints yield%% + projected full cost/time, then stops.
cmd = (f'cd {REPO_DIR} && python sft/generate_reasoning_distill.py '
       f'--limit 3000 --workers 48')
print(cmd)
!{cmd}

In [ ]:
# 6. FULL RUN — uncapped: runs until the problem pool is exhausted OR your API
# credits run out (whichever first). Everything verified so far is already saved
# on Drive; if it stops, just re-run this cell to resume (no dupes, no re-spend).
cmd = f'cd {REPO_DIR} && python sft/generate_reasoning_distill.py --workers 48'
print(cmd)
!{cmd}

In [ ]:
# 7. Inspect output
import json, os
base = os.path.join(SYNAPSE_DIR, 'datasets_sft', 'reasoning_distill')
raw = os.path.join(base, 'reasoning_distill_raw.jsonl')
n = sum(1 for _ in open(raw)) if os.path.exists(raw) else 0
print('kept traces:', n)
if n:
    ex = json.loads(open(raw).readline())
    print('\nQ:', ex['messages'][0]['content'][:300])
    print('\nA:', ex['messages'][1]['content'][:600])
mp = os.path.join(base, 'meta_raw.json')
if os.path.exists(mp):
    print('\nmeta:', json.load(open(mp)))

## After generating → fold into SFT
```bash
python sft/tokenize_sft_data.py --datasets reasoning_distill --force
python sft/consolidate_sft_data.py
```
Then add `"reasoning_distill": ~0.28` to `SFT_DATA_MIX` in `sft.py` (lowering the others) and re-run training.

**Notes**
- Resumable: a `checkpoint.json` tracks processed problems — re-run cell 6 if Colab/laptop drops.
- **No cost cap** here — it runs until the pool is done or your API credits run out. Watch the `kept`/`cost` lines; everything is saved per-trace on Drive, so stop anytime and resume.
- **Provider:** DeepSeek direct (`DEEPSEEK_API_KEY`) is used in preference to OpenRouter when both keys are set.
- GSM8K is excluded from the problem pool + decontaminated against, so it stays a clean benchmark.